# 03 · Variables de exposición promocional

Construye las variables que responden la pregunta de la tesis (D‑27): para cada SKU y día, su **propia promoción** y la promoción de los **demás SKUs**, resumida en tres niveles (D‑07):

| Nivel | Variable | Pregunta |
|---|---|---|
| Propio SKU | `onpromotion` y sus bloques de rezagos y adelantos | (i) ¿la promoción aumenta las ventas del SKU, y las canibaliza en los días siguientes? |
| Misma clase | `promo_clase_otros`: proporción de los otros SKUs activos de la clase en promoción | (ii) ¿canibaliza a los SKUs más cercanos? |
| Resto de la categoría | `promo_cat_otras_clases`: proporción de los SKUs activos de las otras clases de la categoría en promoción | (ii) ¿afecta al resto de su categoría? |
| Otras categorías | `p__<FAMILIA>`: proporción del surtido activo de cada categoría en promoción | (iii) ¿afecta a SKUs de otras categorías? |

| | |
|---|---|
| **Entrada** | `panel_t44_sku_dia.parquet` (de `01`), `calendario_t44.csv` (de `02`) |
| **Salidas** | `intensidad_t44_categoria_dia.csv`, `exposicion_t44_categorias_dia.csv`, `exposicion_t44_resto_por_objetivo.csv`, `exposiciones_t44_sku_dia.parquet` |
| **Decisiones que implementa** | D‑06, D‑07, D‑09, D‑14, D‑26, D‑28 |

**Por qué no se estiman efectos SKU a SKU.** Con 3.656 SKUs hay unos 13 millones de pares y solo 590 días: no hay información para estimar cada par. Los tres niveles anteriores resumen la promoción de muchos SKUs en pocas variables. Supone que, dentro de cada grupo, da lo mismo cuál SKU está en promoción; ese supuesto debe declararse en la tesis.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook construye las variables de exposición promocional del SKU, de su clase, de su categoría y de las
# otras categorías, con sus bloques de rezagos y adelantos.
# ------------------------------------------------------------------------------------------------------------
# Implementación: Ramiro Saltos
# Fecha de versión: 2026-10-15
# Revisado: 2026-10-15
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import re

import numpy as np
import pandas as pd

In [2]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
umbral_programa_proporcion = 0.5     # D-26: proporción del surtido en promoción que define un día de programa
umbral_programa_semanas = 0.9        # D-26: fracción de semanas en que debe cumplirse para ser programa fijo
umbral_exposicion = 0.20             # D-06: fracción mínima de días con promoción fuera del programa

# D-28: núcleo de categorías de exposición; las demás categorías fuente se agrupan en "resto"
nucleo_exposicion = [
    "MEATS", "POULTRY", "DELI", "PRODUCE", "DAIRY",
    "BREAD/BAKERY", "BEVERAGES", "LIQUOR,WINE,BEER", "GROCERY I", "CLEANING",
]

# D-14: bloques de días desplazados (positivo = rezago, negativo = adelanto); sin rezago 7
bloques = {
    "l1_3": [1, 2, 3],
    "l4_6": [4, 5, 6],
    "f1_3": [-1, -2, -3],
}

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")
salida_intensidad = os.path.join(ruta_procesados, "intensidad_t44_categoria_dia.csv")
salida_categorias = os.path.join(ruta_procesados, "exposicion_t44_categorias_dia.csv")
salida_resto = os.path.join(ruta_procesados, "exposicion_t44_resto_por_objetivo.csv")
salida_sku = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")

In [3]:
# Leer el panel y el calendario
panel = pd.read_parquet(archivo_panel, columns=["date", "item_nbr", "family", "class", "onpromotion"])
calendario = pd.read_csv(archivo_calendario, usecols=["date", "dow", "ventana_terremoto"], parse_dates=["date"])

# Numerar los días con operación (t = 0, ..., 589) para desplazar por día de operación
calendario = calendario.sort_values("date").reset_index(drop=True)
calendario["t"] = np.arange(len(calendario))

panel = panel.merge(calendario, on="date", how="left").sort_values(["item_nbr", "date"]).reset_index(drop=True)
print(f"Panel: {len(panel):,} SKU-días | días con operación: {len(calendario)}")

Panel: 1,935,758 SKU-días | días con operación: 590


## 1. Intensidad promocional por categoría y día (D‑07)

Para cada categoría y día: SKUs activos (dentro de su ventana de vida), SKUs en promoción, la **proporción del surtido activo en promoción** (`p_prop`, medida principal) y $\log(1+N)$ (`p_log`, robustez). La promoción usa la imputación principal de D‑25.

In [4]:
# Calcular la intensidad promocional por categoría y día
intensidad = (
    panel.groupby(["family", "date"], observed=True)
    .agg(n_activos=("item_nbr", "size"), n_promo=("onpromotion", "sum"))
    .reset_index()
    .merge(calendario[["date", "dow"]], on="date", how="left")
)
intensidad["p_prop"] = intensidad["n_promo"] / intensidad["n_activos"]
intensidad["p_log"] = np.log1p(intensidad["n_promo"])

print(f"Filas categoría-día: {len(intensidad):,}")

Filas categoría-día: 18,551


## 2. Programas semanales fijos (D‑26)

Un **programa semanal fijo** es un par (categoría, día de la semana) en que al menos la mitad del surtido activo está en promoción en al menos el 90% de las semanas. En esos días la promoción de la categoría es indistinguible del efecto del día de la semana, así que su efecto no se puede estimar.

Se crean dos variables:

- `program_promotion` (SKU‑día): el SKU está en promoción **por el programa** de su categoría.
- `p_prop_fuera_programa` (categoría‑día): la intensidad de la categoría fuera de los días de su programa (cero en esos días). Es la que se usa como exposición de las categorías con programa.
- `dia_programa` (categoría‑día y SKU‑día): el día es un día de programa de la categoría. Los modelos de las seis categorías con programa se estiman **sin sus días de programa** (A‑03): en esos días toda la categoría está en promoción, igual todas las semanas, y no aporta información.

In [5]:
def detectar_programas(
    intensidad: pd.DataFrame,
    umbral_proporcion: float,
    umbral_semanas: float,
) -> pd.DataFrame:
    """
    Esta función detecta los programas semanales fijos: pares (categoría, día de la semana) con gran parte del
    surtido en promoción casi todas las semanas.

    :param intensidad:          un DataFrame categoría-día con las columnas family, dow y p_prop.
    :param umbral_proporcion:   proporción mínima del surtido activo en promoción para contar la semana.
    :param umbral_semanas:      fracción mínima de semanas que deben cumplir el umbral anterior.
    :return:                    un DataFrame con las columnas family, dow y fraccion_semanas de los programas
                                detectados.
    """

    # Calcular, para cada categoría y día de la semana, la fracción de semanas sobre el umbral de proporción
    sobre_umbral = (intensidad["p_prop"] >= umbral_proporcion).rename("sobre_umbral")
    fraccion = (
        sobre_umbral.groupby([intensidad["family"], intensidad["dow"]], observed=True)
        .mean()
        .rename("fraccion_semanas")
        .reset_index()
    )

    return fraccion.loc[fraccion["fraccion_semanas"] >= umbral_semanas].reset_index(drop=True)


nombre_dia = {0: "lunes", 1: "martes", 2: "miércoles", 3: "jueves", 4: "viernes", 5: "sábado", 6: "domingo"}

programas = detectar_programas(intensidad, umbral_programa_proporcion, umbral_programa_semanas)
programas["dia"] = programas["dow"].map(nombre_dia)
print(programas.to_string(index=False))

# Verificar los programas encontrados en el diagnóstico del profesor guía (2026-10-15)
esperados = {("MEATS", 4), ("POULTRY", 4), ("SEAFOOD", 4), ("EGGS", 4), ("DELI", 4), ("PRODUCE", 2)}
encontrados = set(zip(programas["family"].astype(str), programas["dow"]))
assert encontrados == esperados, f"programas distintos de los esperados: {encontrados ^ esperados}"

 family  dow  fraccion_semanas       dia
   DELI    4               1.0   viernes
   EGGS    4               1.0   viernes
  MEATS    4               1.0   viernes
POULTRY    4               1.0   viernes
PRODUCE    2               1.0 miércoles
SEAFOOD    4               1.0   viernes


In [6]:
# Marcar los días de programa en la intensidad por categoría
claves = programas[["family", "dow"]].assign(dia_programa=True)
intensidad = intensidad.merge(claves, on=["family", "dow"], how="left")
intensidad["dia_programa"] = intensidad["dia_programa"].astype("boolean").fillna(False).astype(bool)
intensidad["p_prop_fuera_programa"] = intensidad["p_prop"].where(~intensidad["dia_programa"], 0.0)
intensidad["n_promo_fuera_programa"] = intensidad["n_promo"].where(~intensidad["dia_programa"], 0)

# Marcar la promoción de programa en el panel SKU-día
panel = panel.merge(intensidad[["family", "date", "dia_programa"]], on=["family", "date"], how="left")
panel["program_promotion"] = panel["onpromotion"] & panel["dia_programa"]

print(f"SKU-días en promoción:             {panel['onpromotion'].sum():,}")
print(f"  de ellos, por programa fijo:      {panel['program_promotion'].sum():,} "
      f"({panel['program_promotion'].sum() / panel['onpromotion'].sum():.1%})")

SKU-días en promoción:             157,593
  de ellos, por programa fijo:      36,201 (23.0%)


## 3. Exposición dentro de la categoría (preguntas ii)

Para cada SKU‑día, la proporción en promoción de **los otros** SKUs activos de su clase (excluye al propio SKU) y de los SKUs activos de **las otras clases** de su categoría. Quedan sin definir (nulas) cuando no hay otros SKUs activos en el grupo, por ejemplo en las clases con un solo SKU.

In [7]:
# Contar SKUs activos y en promoción por clase y por categoría en cada día
por_clase = panel.groupby(["family", "class", "date"], observed=True)["onpromotion"]
panel["n_activos_clase"] = por_clase.transform("size")
panel["n_promo_clase"] = por_clase.transform("sum")

por_categoria = panel.groupby(["family", "date"], observed=True)["onpromotion"]
panel["n_activos_cat"] = por_categoria.transform("size")
panel["n_promo_cat"] = por_categoria.transform("sum")

# Calcular la proporción en promoción de los otros SKUs de la clase (sin el propio SKU)
otros_clase = panel["n_activos_clase"] - 1
panel["promo_clase_otros"] = (
    (panel["n_promo_clase"] - panel["onpromotion"].astype(int)) / otros_clase
).where(otros_clase > 0)

# Calcular la proporción en promoción de los SKUs de las otras clases de la categoría
otras_clases = panel["n_activos_cat"] - panel["n_activos_clase"]
panel["promo_cat_otras_clases"] = (
    (panel["n_promo_cat"] - panel["n_promo_clase"]) / otras_clases
).where(otras_clases > 0)

print(panel[["promo_clase_otros", "promo_cat_otras_clases"]].describe().round(3).to_string())

       promo_clase_otros  promo_cat_otras_clases
count        1895579.000             1904896.000
mean               0.082                   0.082
std                0.166                   0.135
min                0.000                   0.000
25%                0.000                   0.027
50%                0.016                   0.058
75%                0.092                   0.095
max                1.000                   1.000


## 4. Rezagos y adelantos en bloques (D‑14, D‑09)

Cada bloque es el **promedio** de la variable en los días de operación desplazados: `l1_3` = días 1 a 3 anteriores, `l4_6` = días 4 a 6 anteriores, `f1_3` = días 1 a 3 siguientes (adelantos, para la prueba placebo). No se usa el rezago 7: en las categorías con programa semanal es casi igual a la promoción del mismo día.

Reglas para los días que faltan:

- **Fuera del período** (antes del 2016‑01‑02 o después del 2017‑08‑15): nulo.
- **Ventana del terremoto** (D‑09): nulo. Así, un bloque que toca esa ventana queda nulo y la fila sale de la estimación.
- **Fuera de la ventana de vida del SKU** (todavía no se vendía o ya no se vendía): para la promoción propia se supone cero; para las exposiciones de clase y categoría, nulo.

El desplazamiento es por **día de operación**: el día anterior al 26‑12‑2016 es el 24‑12‑2016, porque la tienda no abre el 25.

In [8]:
def agregar_bloques_sku(
    panel: pd.DataFrame,
    columna: str,
    bloques: dict,
    relleno_fuera_ventana: float,
) -> pd.DataFrame:
    """
    Esta función agrega al panel SKU-día el promedio de una variable en bloques de días de operación desplazados.

    :param panel:                   un DataFrame ordenado por SKU y fecha, con las columnas item_nbr, t (índice del día
                                    de operación) y ventana_terremoto.
    :param columna:                 el nombre de la variable a desplazar.
    :param bloques:                 un diccionario {sufijo: lista de desplazamientos}; positivo = rezago, negativo =
                                    adelanto.
    :param relleno_fuera_ventana:   el valor para los días fuera de la ventana de vida del SKU (0.0 o np.nan).
    :return:                        el mismo DataFrame con una columna nueva '<columna>_<sufijo>' por bloque.
    """

    # Anular los días de la ventana del terremoto para que no entren a ningún bloque
    valores = panel[columna].astype(float).where(~panel["ventana_terremoto"])
    por_sku = valores.groupby(panel["item_nbr"])

    # Posición de cada fila dentro de la ventana de vida del SKU, desde el inicio y desde el final
    pos_inicio = panel.groupby("item_nbr").cumcount()
    pos_final = panel.groupby("item_nbr").cumcount(ascending=False)
    ultimo_t = panel["t"].max()

    for sufijo, desplazamientos in bloques.items():
        desplazadas = []
        for k in desplazamientos:
            # Desplazar k días de operación dentro del SKU
            s = por_sku.shift(k)

            # Rellenar fuera de la ventana de vida y anular fuera del período
            if k > 0:
                s = s.where(pos_inicio >= k, relleno_fuera_ventana).where(panel["t"] - k >= 0)
            else:
                s = s.where(pos_final >= -k, relleno_fuera_ventana).where(panel["t"] - k <= ultimo_t)
            desplazadas.append(s)

        # Promediar el bloque; si falta algún día, el bloque queda nulo
        panel[f"{columna}_{sufijo}"] = pd.concat(desplazadas, axis=1).mean(axis=1, skipna=False).astype("float32")

    return panel


# Verificar que el panel esté ordenado por SKU y fecha, con días de operación consecutivos dentro de cada SKU
assert panel["item_nbr"].is_monotonic_increasing, "el panel debe estar ordenado por SKU"
assert (panel.groupby("item_nbr")["t"].diff().dropna() == 1).all(), "faltan días dentro de la ventana de algún SKU"

panel = agregar_bloques_sku(panel, "onpromotion", bloques, relleno_fuera_ventana=0.0)
panel = agregar_bloques_sku(panel, "promo_clase_otros", bloques, relleno_fuera_ventana=np.nan)
panel = agregar_bloques_sku(panel, "promo_cat_otras_clases", bloques, relleno_fuera_ventana=np.nan)

columnas_bloques = [c for c in panel.columns if c.endswith(tuple(bloques))]
print(panel[columnas_bloques].isna().mean().mul(100).round(1).rename("% nulos").to_string())

onpromotion_l1_3                8.3
onpromotion_l4_6                8.7
onpromotion_f1_3                8.3
promo_clase_otros_l1_3         10.3
promo_clase_otros_l4_6         10.8
promo_clase_otros_f1_3         10.3
promo_cat_otras_clases_l1_3     9.9
promo_cat_otras_clases_l4_6    10.4
promo_cat_otras_clases_f1_3     9.9


## 5. Exposición a las otras categorías (pregunta iii; D‑06)

Una tabla con **una fila por día** y una columna por categoría con la intensidad que se usa como exposición: `p_prop` para las categorías sin programa y `p_prop_fuera_programa` para las que tienen programa. Solo entran como fuente de exposición las categorías con promoción fuera de su programa en al menos el 20% de los días (D‑06); las demás pueden estudiarse como categoría objetivo, pero no aportan variación como tratamiento.

In [9]:
def nombre_columna(familia: str) -> str:
    """
    Esta función convierte el nombre de una familia en un nombre de columna sin espacios ni signos.

    :param familia:     el nombre de la familia (p. ej., 'LIQUOR,WINE,BEER').
    :return:            el nombre de columna (p. ej., 'p__LIQUOR_WINE_BEER').
    """

    return "p__" + re.sub(r"[^A-Z0-9]+", "_", familia.upper()).strip("_")


# Calcular la fracción de días con promoción fuera del programa por categoría
ancho = (
    intensidad.pivot(index="date", columns="family", values="p_prop_fuera_programa")
    .reindex(calendario["date"])
    .fillna(0.0)
)
fraccion_dias = (ancho > 0).mean().sort_values(ascending=False)
familias_exposicion = fraccion_dias.index[fraccion_dias >= umbral_exposicion].tolist()

print(f"Categorías fuente de exposición ({len(familias_exposicion)} de {len(fraccion_dias)}):")
print(fraccion_dias.round(2).to_string())

# Conservar las categorías fuente y renombrar las columnas
ancho = ancho[familias_exposicion]
ancho.columns = [nombre_columna(str(f)) for f in ancho.columns]
ancho.columns.name = None

Categorías fuente de exposición (22 de 32):
family
GROCERY I                     1.00
BEVERAGES                     0.99
CLEANING                      0.99
HOME AND KITCHEN II           0.97
PERSONAL CARE                 0.96
DAIRY                         0.95
HOME CARE                     0.90
BREAD/BAKERY                  0.85
FROZEN FOODS                  0.84
DELI                          0.77
LIQUOR,WINE,BEER              0.68
PRODUCE                       0.67
HOME AND KITCHEN I            0.58
BEAUTY                        0.53
SEAFOOD                       0.47
EGGS                          0.41
PREPARED FOODS                0.41
MEATS                         0.39
AUTOMOTIVE                    0.32
PET SUPPLIES                  0.27
POULTRY                       0.26
SCHOOL AND OFFICE SUPPLIES    0.22
LINGERIE                      0.15
GROCERY II                    0.14
CELEBRATION                   0.11
LAWN AND GARDEN               0.07
PLAYERS AND ELECTRONICS       0.06
LADI

In [10]:
def agregar_bloques_diarios(ancho: pd.DataFrame, bloques: dict, en_ventana: pd.Series) -> pd.DataFrame:
    """
    Esta función agrega a una tabla diaria el promedio de cada columna en bloques de días de operación desplazados.

    :param ancho:       un DataFrame con una fila por día de operación, ordenado por fecha.
    :param bloques:     un diccionario {sufijo: lista de desplazamientos}; positivo = rezago, negativo = adelanto.
    :param en_ventana:  una serie booleana alineada con 'ancho' que marca la ventana del terremoto.
    :return:            un DataFrame con las columnas originales y una columna '<columna>_<sufijo>' por bloque.
    """

    # Anular la ventana del terremoto antes de desplazar
    base = ancho.copy()
    base.loc[en_ventana.to_numpy()] = np.nan
    resultado = [ancho]

    for sufijo, desplazamientos in bloques.items():
        # Promediar los días desplazados; fuera del período o en la ventana del terremoto, el bloque queda nulo
        suma = sum(base.shift(k) for k in desplazamientos)
        bloque = (suma / len(desplazamientos)).add_suffix(f"_{sufijo}")
        resultado.append(bloque)

    return pd.concat(resultado, axis=1).astype("float32")


exposicion_categorias = agregar_bloques_diarios(ancho, bloques, calendario["ventana_terremoto"])
exposicion_categorias = exposicion_categorias.reset_index()
print(f"Tabla diaria: {exposicion_categorias.shape[0]} días x {exposicion_categorias.shape[1] - 1} columnas")

Tabla diaria: 590 días x 88 columnas


## 6. Núcleo de categorías y variable "resto" (D‑28)

Con las 22 categorías fuente, cada modelo tendría demasiados efectos entre categorías para 544 días. La especificación principal usa un **núcleo de 10 categorías** (parámetro `nucleo_exposicion`) y una variable `p__RESTO` que agrupa a las demás categorías fuente: la proporción del surtido activo de esas categorías en promoción fuera de sus programas.

`p__RESTO` pondera por número de SKUs, así que pesan más las categorías grandes del resto (PERSONAL CARE, HOME CARE). El resto funciona como **control**: no se interpreta, y los resultados se enfocan en las categorías del núcleo. Como robustez basta con estimar el modelo sin `p__RESTO` (A‑09).

`p__RESTO` depende de la categoría objetivo: si la categoría que se modela pertenece al resto, su propia promoción no debe estar en esa variable. Por eso se guarda una tabla con una fila por **día y categoría objetivo**. Las 22 columnas individuales quedan en `exposicion_t44_categorias_dia.csv` para el análisis de robustez.

In [11]:
def calcular_resto(intensidad: pd.DataFrame, familias: list, fechas: pd.Series) -> pd.DataFrame:
    """
    Esta función calcula, para cada día, la proporción del surtido activo de un grupo de categorías que está en
    promoción fuera de sus programas.

    :param intensidad:  un DataFrame categoría-día con las columnas family, date, n_activos y n_promo_fuera_programa.
    :param familias:    la lista de categorías que forman el grupo.
    :param fechas:      la serie de días con operación, ordenada.
    :return:            un DataFrame con una fila por día (índice date) y la columna p__RESTO.
    """

    # Sumar SKUs activos y SKUs en promoción fuera del programa de todas las categorías del grupo
    grupo = intensidad.loc[intensidad["family"].isin(familias)]
    sumas = grupo.groupby("date")[["n_activos", "n_promo_fuera_programa"]].sum().reindex(fechas)

    # Calcular la proporción del grupo; un día sin SKUs activos en el grupo cuenta como sin promoción
    resto = (sumas["n_promo_fuera_programa"] / sumas["n_activos"]).fillna(0.0)
    return resto.rename("p__RESTO").to_frame()


# Verificar que el núcleo esté dentro de las categorías fuente de exposición
fuera_de_fuentes = set(nucleo_exposicion) - set(map(str, familias_exposicion))
assert not fuera_de_fuentes, f"categorías del núcleo que no son fuente de exposición: {fuera_de_fuentes}"
fuentes_resto = [str(f) for f in familias_exposicion if str(f) not in nucleo_exposicion]
print(f"Núcleo ({len(nucleo_exposicion)}): {nucleo_exposicion}")
print(f"Resto ({len(fuentes_resto)}): {fuentes_resto}")

# Calcular el resto para cada categoría objetivo, excluyendo a la propia categoría del grupo
tablas_resto = []
for objetivo in sorted(map(str, intensidad["family"].unique())):
    grupo = [f for f in fuentes_resto if f != objetivo]
    resto = calcular_resto(intensidad, grupo, calendario["date"])
    resto = agregar_bloques_diarios(resto, bloques, calendario["ventana_terremoto"]).reset_index()
    tablas_resto.append(resto.assign(family=objetivo))

exposicion_resto = pd.concat(tablas_resto, ignore_index=True)
exposicion_resto = exposicion_resto[["family", "date"] + [c for c in exposicion_resto.columns if c.startswith("p__")]]
print(f"Tabla del resto: {len(exposicion_resto):,} filas (día x categoría objetivo)")

Núcleo (10): ['MEATS', 'POULTRY', 'DELI', 'PRODUCE', 'DAIRY', 'BREAD/BAKERY', 'BEVERAGES', 'LIQUOR,WINE,BEER', 'GROCERY I', 'CLEANING']
Resto (12): ['HOME AND KITCHEN II', 'PERSONAL CARE', 'HOME CARE', 'FROZEN FOODS', 'HOME AND KITCHEN I', 'BEAUTY', 'SEAFOOD', 'EGGS', 'PREPARED FOODS', 'AUTOMOTIVE', 'PET SUPPLIES', 'SCHOOL AND OFFICE SUPPLIES']
Tabla del resto: 18,880 filas (día x categoría objetivo)


## 7. Exportación

In [12]:
# Guardar la intensidad por categoría-día
columnas_intensidad = [
    "family", "date", "dow", "n_activos", "n_promo", "p_prop", "p_log",
    "dia_programa", "p_prop_fuera_programa", "n_promo_fuera_programa",
]
intensidad[columnas_intensidad].to_csv(salida_intensidad, index=False)

# Guardar la exposición diaria a las categorías y la del resto en CSV (tablas pequeñas, se abren en Excel; A-07)
exposicion_categorias.to_csv(salida_categorias, index=False)
exposicion_resto.to_csv(salida_resto, index=False)

# Guardar las exposiciones SKU-día en Parquet: 1,9 millones de filas no caben en Excel (A-07)
columnas_sku = [
    "item_nbr", "date", "dia_programa", "program_promotion", "promo_clase_otros", "promo_cat_otras_clases",
] + columnas_bloques
exposiciones_sku = panel[columnas_sku]
assert len(exposiciones_sku) == len(panel) and not exposiciones_sku.duplicated(["item_nbr", "date"]).any(), \
    "las exposiciones deben tener una fila por SKU-día del panel"
exposiciones_sku.to_parquet(salida_sku, index=False)

for archivo in [salida_intensidad, salida_categorias, salida_resto, salida_sku]:
    print(f"Guardado: {archivo}  ({os.path.getsize(archivo) / 1024**2:.1f} MB)")

print(f"\n>>>> Proceso finalizado <<<<")

Guardado: ..\data\procesados\intensidad_t44_categoria_dia.csv  (1.3 MB)
Guardado: ..\data\procesados\exposicion_t44_categorias_dia.csv  (0.4 MB)
Guardado: ..\data\procesados\exposicion_t44_resto_por_objetivo.csv  (1.2 MB)
Guardado: ..\data\procesados\exposiciones_t44_sku_dia.parquet  (15.3 MB)

>>>> Proceso finalizado <<<<


## Cómo usar estas tablas en los modelos (fases 1 y 2)

La base de estimación para una categoría objetivo $j$ se arma así:

1. Tomar las filas de `panel_t44_sku_dia.parquet` de la categoría $j$.
2. Unir `exposiciones_t44_sku_dia.parquet` por `item_nbr` y `date`.
3. Unir `calendario_t44.csv` por `date`.
4. Unir de `exposicion_t44_categorias_dia.csv`, por `date`, **solo las columnas del núcleo** (D‑28), sin las de la propia categoría $j$ si pertenece al núcleo (su promoción ya está en los niveles propio, clase y categoría).
5. Unir `exposicion_t44_resto_por_objetivo.csv` por `date` y `family` = $j$ (la variable `p__RESTO` ya excluye a la categoría $j$).
6. Si $j$ es una de las seis categorías con programa (MEATS, POULTRY, SEAFOOD, EGGS, DELI, PRODUCE), excluir las filas con `dia_programa` (A‑03).
7. Excluir los días de la ventana del terremoto (`ventana_terremoto`) y las filas con bloques nulos.

Recuerden que la exposición a otras categorías varía solo por día: los errores estándar deben agruparse por SKU y por semana (D‑15), y la información efectiva para esos efectos son los días, no las filas. La especificación de los modelos está en `metodologia.tex`, sección "Modelos".